# Chapter 8: Estimation
**Spine:** Think Stats by Allen Downey (adapted for applied analytics work)  
**Libraries:** Production-grade only — `numpy`, `pandas`, `scipy.stats`, `matplotlib`, `seaborn`

---
## Why estimation matters

Everything in Chapters 3-7 was **descriptive** — we described what our sample looks like.
Estimation is **inferential** — we use our sample to make claims about the population.

This is where statistics becomes directly useful in analytical work:
- What is the true average income in California, given our sample?
- How confident are we in that estimate?
- How large a sample do we need to achieve a desired precision?
- Did our intervention actually change the outcome, or was it random variation?

**Topics covered:**
1. Point estimation and estimator properties — bias, variance, MSE
2. Sampling distribution and standard error
3. Confidence intervals — analytical (Z and t)
4. Bootstrap resampling — the production workhorse
5. Maximum Likelihood Estimation formalized
6. Estimation in practice — margin of error, sample size, A/B testing foundation

In [ ]:
import ssl
import certifi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import fetch_california_housing

ssl._create_default_https_context = ssl.create_default_context(cafile=certifi.where())

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

np.random.seed(42)

housing = fetch_california_housing(as_frame=True).frame
tips    = sns.load_dataset('tips')

income    = housing['MedInc'].values
house_val = housing['MedHouseVal'].values
total_bill = tips['total_bill'].values
tip        = tips['tip'].values

print('California Housing:', housing.shape)
print('Tips:              ', tips.shape)

---
## 1. Point estimation and estimator properties

A **point estimator** is a function of the sample that produces a single value as an
estimate of a population parameter.

Common estimators:
- Sample mean $\bar{x}$ estimates population mean $\mu$
- Sample variance $s^2$ estimates population variance $\sigma^2$
- Sample proportion $\hat{p}$ estimates population proportion $p$

### Bias

An estimator $\hat{\theta}$ is **unbiased** if its expected value equals the true parameter:

$$\text{Bias}(\hat{\theta}) = E[\hat{\theta}] - \theta = 0$$

### Variance of an estimator

How much does the estimator fluctuate across different samples?

$$\text{Var}(\hat{\theta}) = E\left[(\hat{\theta} - E[\hat{\theta}])^2\right]$$

### Mean Squared Error — the unified measure

MSE trades off bias and variance into a single number:

$$MSE(\hat{\theta}) = \text{Bias}^2(\hat{\theta}) + \text{Var}(\hat{\theta})$$

This decomposition reappears in ML model evaluation — the bias-variance tradeoff
in regularization, cross-validation, and model selection all derive from this identity.

In [ ]:
# Demonstrate bias in variance estimation
# The classic example: sample variance with n vs n-1
# Population: treat full California Housing income as the true population
# Repeatedly sample from it and compare biased vs unbiased variance estimates

true_variance = income.var()  # population variance (ddof=0)
n_simulations = 10000
sample_size   = 30

biased_estimates   = []
unbiased_estimates = []

for _ in range(n_simulations):
    sample = np.random.choice(income, size=sample_size, replace=True)
    biased_estimates.append(sample.var(ddof=0))    # divide by n
    unbiased_estimates.append(sample.var(ddof=1))  # divide by n-1

biased_estimates   = np.array(biased_estimates)
unbiased_estimates = np.array(unbiased_estimates)

print('--- Bias demonstration: sample variance ---')
print(f'True population variance:           {true_variance:.4f}')
print(f'Mean of biased estimates (ddof=0):   {biased_estimates.mean():.4f}  bias = {biased_estimates.mean() - true_variance:+.4f}')
print(f'Mean of unbiased estimates (ddof=1): {unbiased_estimates.mean():.4f}  bias = {unbiased_estimates.mean() - true_variance:+.4f}')
print()
print('ddof=0 systematically underestimates — biased downward.')
print('ddof=1 (Bessel correction) removes the bias.')
print()
print('Why ddof=0 is biased:')
print('  The sample mean is computed from the same data — it is already')
print('  closer to the sample than the true population mean.')
print('  Deviations from the sample mean are systematically smaller')
print('  than deviations from the true mean. Dividing by n-1 corrects this.')

In [ ]:
# MSE demonstration — bias vs variance tradeoff
# Compare three estimators of the population mean:
# 1. Sample mean — unbiased, moderate variance
# 2. Sample median — slightly biased for skewed data, lower variance
# 3. Deliberately biased estimator — mean * 0.9

true_mean = income.mean()
n_sim     = 10000
n         = 50

means   = []
medians = []
biased  = []

for _ in range(n_sim):
    s = np.random.choice(income, size=n, replace=True)
    means.append(s.mean())
    medians.append(np.median(s))
    biased.append(s.mean() * 0.9)

means   = np.array(means)
medians = np.array(medians)
biased  = np.array(biased)

def mse(estimates, true_value):
    return np.mean((estimates - true_value) ** 2)

def bias(estimates, true_value):
    return estimates.mean() - true_value

print('--- MSE decomposition: Bias² + Variance ---')
print(f'True mean: {true_mean:.4f}')
print()
for name, est in [('Sample mean', means), ('Sample median', medians), ('Biased (mean*0.9)', biased)]:
    b   = bias(est, true_mean)
    v   = est.var()
    m   = mse(est, true_mean)
    print(f'{name:<22}  bias={b:+.4f}  var={v:.4f}  mse={m:.4f}  (bias²+var={b**2+v:.4f})')

In [ ]:
# Visualize sampling distributions of the three estimators
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, est, color) in zip(axes, [
    ('Sample mean',        means,   'steelblue'),
    ('Sample median',      medians, 'darkorange'),
    ('Biased (mean*0.9)',  biased,  'crimson')
]):
    ax.hist(est, bins=60, density=True, color=color, alpha=0.7)
    ax.axvline(x=true_mean,   color='black',  linestyle='-',  linewidth=2, label=f'True mean = {true_mean:.3f}')
    ax.axvline(x=est.mean(),  color='crimson', linestyle='--', linewidth=1.5, label=f'Est mean = {est.mean():.3f}')
    ax.set_title(f'{name}\nMSE = {mse(est, true_mean):.4f}')
    ax.set_xlabel('Estimate value')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Sampling distributions — bias vs variance tradeoff', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('estimator_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Sampling distribution and standard error

The **sampling distribution** of an estimator is the distribution of that estimator
across all possible samples of size $n$ from the population.

The **standard error** is the standard deviation of the sampling distribution:

$$SE(\bar{x}) = \frac{\sigma}{\sqrt{n}}$$

It measures how precise your estimate is — smaller SE means more precise.

**Key insight:** SE shrinks as $\sqrt{n}$ — to halve your SE you need 4x the data.

In [ ]:
# Standard error — analytical vs empirical
sample_sizes  = [10, 30, 100, 500, 1000]
n_simulations = 5000
sigma         = income.std()  # population std

results = []
for n in sample_sizes:
    sample_means = [
        np.random.choice(income, size=n, replace=True).mean()
        for _ in range(n_simulations)
    ]
    sample_means  = np.array(sample_means)
    se_analytical = sigma / np.sqrt(n)
    se_empirical  = sample_means.std()
    results.append({
        'n':            n,
        'se_analytical': round(se_analytical, 4),
        'se_empirical':  round(se_empirical,  4),
        'difference':    round(abs(se_analytical - se_empirical), 4)
    })

results_df = pd.DataFrame(results)
print('--- Standard error: analytical vs empirical ---')
print(results_df.to_string(index=False))
print()
print('Analytical and empirical SE converge as n grows.')
print('To halve SE you need 4x the sample size — sqrt(n) relationship.')

In [ ]:
# SE for non-mean statistics — bootstrap is required
# Standard error of the median and correlation have no simple closed-form formula
n_boot = 5000
sample = np.random.choice(income, size=200, replace=True)

boot_means   = [np.random.choice(sample, size=len(sample), replace=True).mean()   for _ in range(n_boot)]
boot_medians = [np.median(np.random.choice(sample, size=len(sample), replace=True)) for _ in range(n_boot)]

boot_means   = np.array(boot_means)
boot_medians = np.array(boot_medians)

print('--- Standard error via bootstrap (n=200 sample) ---')
print(f'SE of mean   (analytical): {sample.std() / np.sqrt(len(sample)):.4f}')
print(f'SE of mean   (bootstrap):  {boot_means.std():.4f}')
print(f'SE of median (bootstrap):  {boot_medians.std():.4f}  (no closed-form formula)')
print()
print('Bootstrap gives SE for any statistic — not just the mean.')
print('This is why it is the production workhorse for estimation.')

---
## 3. Confidence intervals — analytical

A **confidence interval** gives a range of plausible values for the population parameter.

### The correct interpretation

A 95% CI does **not** mean: *"there is a 95% probability the true parameter is in this interval."*

The true parameter is fixed — it is either in the interval or it is not.

The correct interpretation: *"if we repeated this sampling procedure many times and computed
a CI each time, 95% of those intervals would contain the true parameter."*

### Z-interval (large samples, known $\sigma$)

$$\bar{x} \pm z_{\alpha/2} \cdot \frac{\sigma}{\sqrt{n}}$$

For 95% CI: $z_{0.025} = 1.96$

### t-interval (small samples, unknown $\sigma$)

$$\bar{x} \pm t_{\alpha/2, n-1} \cdot \frac{s}{\sqrt{n}}$$

The t-distribution has heavier tails than normal — accounting for the additional
uncertainty of estimating $\sigma$ from a small sample. As $n \to \infty$, the
t-distribution converges to the normal distribution.

In [ ]:
# Confidence intervals — Z vs t
sample_small = np.random.choice(income, size=20,   replace=True)  # small sample
sample_large = np.random.choice(income, size=1000, replace=True)  # large sample
confidence   = 0.95

def z_interval(sample, confidence=0.95):
    """Z-interval using population std (approximated by sample std for large n)."""
    n    = len(sample)
    mean = sample.mean()
    se   = sample.std(ddof=1) / np.sqrt(n)
    z    = stats.norm.ppf((1 + confidence) / 2)
    return mean - z * se, mean + z * se

def t_interval(sample, confidence=0.95):
    """t-interval — correct for any sample size, unknown population std."""
    n    = len(sample)
    mean = sample.mean()
    se   = sample.std(ddof=1) / np.sqrt(n)
    t    = stats.t.ppf((1 + confidence) / 2, df=n-1)
    return mean - t * se, mean + t * se

# scipy convenience function
def scipy_t_interval(sample, confidence=0.95):
    return stats.t.interval(confidence, df=len(sample)-1,
                            loc=sample.mean(),
                            scale=stats.sem(sample))

print('--- 95% Confidence intervals ---')
print(f'True population mean: {income.mean():.4f}')
print()
print(f'Small sample (n=20):')
lo, hi = z_interval(sample_small)
print(f'  Z-interval:  ({lo:.4f}, {hi:.4f})  width = {hi-lo:.4f}')
lo, hi = t_interval(sample_small)
print(f'  t-interval:  ({lo:.4f}, {hi:.4f})  width = {hi-lo:.4f}')
print(f'  t is wider — accounts for uncertainty in estimating sigma from n=20')
print()
print(f'Large sample (n=1000):')
lo, hi = z_interval(sample_large)
print(f'  Z-interval:  ({lo:.4f}, {hi:.4f})  width = {hi-lo:.4f}')
lo, hi = t_interval(sample_large)
print(f'  t-interval:  ({lo:.4f}, {hi:.4f})  width = {hi-lo:.4f}')
print(f'  Z and t converge at large n — t approaches normal distribution')

In [ ]:
# Demonstrate correct CI interpretation — coverage probability
# If 95% CI is correctly constructed, ~95% of intervals should contain the true mean
true_mean   = income.mean()
n_intervals = 1000
n_per_sample = 30

contains_true_mean = []
intervals          = []

for _ in range(n_intervals):
    s    = np.random.choice(income, size=n_per_sample, replace=True)
    lo, hi = t_interval(s, confidence=0.95)
    contains = lo <= true_mean <= hi
    contains_true_mean.append(contains)
    intervals.append((lo, hi))

coverage = np.mean(contains_true_mean)
print(f'--- Coverage probability demonstration ---')
print(f'Expected coverage: 95.00%')
print(f'Actual coverage:   {coverage*100:.2f}%  ({int(coverage*n_intervals)}/{n_intervals} intervals contain true mean)')
print()
print('This is what 95% CI means: if repeated 1000 times, ~950 intervals contain the truth.')
print('It does NOT mean: the true value has a 95% probability of being in this specific interval.')

# Plot first 50 intervals
fig, ax = plt.subplots(figsize=(12, 6))
for i, ((lo, hi), contains) in enumerate(zip(intervals[:50], contains_true_mean[:50])):
    color = 'steelblue' if contains else 'crimson'
    ax.plot([lo, hi], [i, i], color=color, linewidth=1.5, alpha=0.8)
ax.axvline(x=true_mean, color='black', linewidth=2, label=f'True mean = {true_mean:.3f}')
ax.set_title(f'First 50 CIs — blue contains true mean, red does not (coverage = {coverage*100:.1f}%)')
ax.set_xlabel('Median income ($10,000s)')
ax.set_ylabel('Sample index')
ax.legend()
plt.tight_layout()
plt.savefig('ci_coverage.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Bootstrap resampling — the production workhorse

Bootstrap resampling builds the sampling distribution empirically by resampling
**with replacement** from your observed data — no distributional assumptions required.

**The key idea:** treat your sample as if it were the population, and simulate the
sampling process by drawing from it repeatedly.

**Why it works:** if your sample is representative of the population, then resampling
from it approximates resampling from the population.

**The algorithm:**
1. Draw $n$ observations with replacement from your sample (same size as original)
2. Compute the statistic of interest on this bootstrap sample
3. Repeat $B$ times (typically $B = 1000$ to $10000$)
4. The distribution of bootstrap statistics approximates the sampling distribution

**Bootstrap CI (percentile method):**
For a 95% CI, take the 2.5th and 97.5th percentiles of the bootstrap distribution:

$$CI_{95\%} = [\hat{\theta}^*_{0.025},\ \hat{\theta}^*_{0.975}]$$

In [ ]:
def bootstrap_ci(data: np.ndarray, statistic, n_boot: int = 5000,
                  confidence: float = 0.95) -> tuple:
    """
    Compute bootstrap confidence interval for any statistic.
    Production-safe: uses only numpy.

    Parameters
    ----------
    data       : observed sample
    statistic  : callable — function that takes an array and returns a scalar
    n_boot     : number of bootstrap resamples
    confidence : confidence level (default 0.95)

    Returns
    -------
    (lower, upper, bootstrap_distribution)
    """
    n              = len(data)
    boot_stats     = np.array([
        statistic(data[np.random.randint(0, n, size=n)])
        for _ in range(n_boot)
    ])
    alpha = 1 - confidence
    lo    = np.percentile(boot_stats, 100 * alpha / 2)
    hi    = np.percentile(boot_stats, 100 * (1 - alpha / 2))
    return lo, hi, boot_stats


# Bootstrap CI for mean — compare to analytical t-interval
sample = np.random.choice(income, size=200, replace=True)

lo_boot, hi_boot, boot_means = bootstrap_ci(sample, np.mean)
lo_t,    hi_t                = t_interval(sample, confidence=0.95)

print('--- Bootstrap vs analytical t-interval (n=200, 95% CI) ---')
print(f'Sample mean:          {sample.mean():.4f}')
print(f'Bootstrap CI:         ({lo_boot:.4f}, {hi_boot:.4f})  width = {hi_boot-lo_boot:.4f}')
print(f'Analytical t-interval:({lo_t:.4f},    {hi_t:.4f})   width = {hi_t-lo_t:.4f}')
print()
print('They agree closely — bootstrap is correctly approximating the sampling distribution.')

In [ ]:
# Bootstrap CI for non-standard statistics
# Median and correlation have no simple closed-form CI formula
sample_tips_bill = np.random.choice(total_bill, size=100, replace=True)
sample_tips_tip  = tip[:100]

# Bootstrap CI for median
lo_med, hi_med, _ = bootstrap_ci(sample_tips_bill, np.median)

# Bootstrap CI for correlation — requires paired bootstrap
n = len(sample_tips_bill)
boot_corrs = []
for _ in range(5000):
    idx = np.random.randint(0, n, size=n)
    r, _ = stats.pearsonr(sample_tips_bill[idx], sample_tips_tip[idx])
    boot_corrs.append(r)
boot_corrs = np.array(boot_corrs)
lo_corr = np.percentile(boot_corrs, 2.5)
hi_corr = np.percentile(boot_corrs, 97.5)

print('--- Bootstrap CI for non-standard statistics ---')
print(f'Median total bill:  {np.median(sample_tips_bill):.4f}')
print(f'95% CI for median:  ({lo_med:.4f}, {hi_med:.4f})')
print()
r_obs, _ = stats.pearsonr(sample_tips_bill, sample_tips_tip)
print(f'Pearson correlation: {r_obs:.4f}')
print(f'95% CI for r:        ({lo_corr:.4f}, {hi_corr:.4f})')
print()
print('Bootstrap gives CI for any statistic without requiring a formula.')
print('This is its primary advantage over analytical methods.')

In [ ]:
# Visualize bootstrap distribution
lo_m, hi_m, boot_dist = bootstrap_ci(sample, np.mean, n_boot=5000)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(boot_dist, bins=60, density=True, color='steelblue', alpha=0.7)
ax.axvline(x=lo_m, color='crimson', linestyle='--', linewidth=2, label=f'2.5th pct = {lo_m:.4f}')
ax.axvline(x=hi_m, color='crimson', linestyle='--', linewidth=2, label=f'97.5th pct = {hi_m:.4f}')
ax.axvline(x=sample.mean(), color='black', linewidth=2, label=f'Observed mean = {sample.mean():.4f}')
ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 5],
                  lo_m, hi_m, alpha=0.15, color='crimson', label='95% CI')
ax.set_title('Bootstrap distribution of sample mean — 5000 resamples')
ax.set_xlabel('Bootstrap sample mean')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.savefig('bootstrap_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Maximum Likelihood Estimation formalized

MLE finds the parameter values that make the observed data **most probable**.

### The likelihood function

Given data $x_1, x_2, ..., x_n$ and a distribution with parameter $\theta$,
the likelihood is the probability of observing this data under $\theta$:

$$L(\theta; x_1,...,x_n) = \prod_{i=1}^{n} f(x_i; \theta)$$

### Log-likelihood

We maximize the log-likelihood instead — products become sums, which are
easier to differentiate and numerically stable:

$$\ell(\theta) = \log L(\theta) = \sum_{i=1}^{n} \log f(x_i; \theta)$$

### MLE for the exponential distribution — analytical derivation

$$\ell(\lambda) = \sum_{i=1}^{n} \log(\lambda e^{-\lambda x_i}) = n\log\lambda - \lambda\sum x_i$$

Taking the derivative and setting to zero:

$$\frac{d\ell}{d\lambda} = \frac{n}{\lambda} - \sum x_i = 0$$

$$\hat{\lambda}_{MLE} = \frac{n}{\sum x_i} = \frac{1}{\bar{x}}$$

The MLE of $\lambda$ is simply the reciprocal of the sample mean — intuitive and elegant.

In [ ]:
# MLE for exponential — analytical vs numerical vs scipy
np.random.seed(42)
true_lambda   = 0.5
exp_sample    = np.random.exponential(scale=1/true_lambda, size=500)

# Analytical MLE
lambda_analytical = 1 / exp_sample.mean()

# Numerical MLE — maximize log-likelihood directly
from scipy.optimize import minimize_scalar

def neg_log_likelihood(lam):
    """Negative log-likelihood for exponential — minimize = maximize likelihood."""
    if lam <= 0:
        return np.inf
    return -(len(exp_sample) * np.log(lam) - lam * exp_sample.sum())

result         = minimize_scalar(neg_log_likelihood, bounds=(0.01, 5), method='bounded')
lambda_numerical = result.x

# scipy MLE
loc_s, scale_s   = stats.expon.fit(exp_sample, floc=0)
lambda_scipy     = 1 / scale_s

print('--- MLE for exponential distribution ---')
print(f'True lambda:        {true_lambda:.4f}')
print(f'Analytical MLE:     {lambda_analytical:.4f}  (1 / sample mean)')
print(f'Numerical MLE:      {lambda_numerical:.4f}  (scipy.optimize)')
print(f'scipy dist.fit():   {lambda_scipy:.4f}  (what Chapter 5 was doing internally)')
print()
print('All three agree — confirming the derivation is correct.')
print('scipy.stats.expon.fit() is running numerical MLE under the hood.')

In [ ]:
# Visualize the log-likelihood function
lambdas    = np.linspace(0.1, 1.5, 500)
log_likes  = [-(len(exp_sample) * np.log(l) - l * exp_sample.sum()) * -1
              for l in lambdas]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lambdas, log_likes, color='steelblue', linewidth=2)
ax.axvline(x=lambda_analytical, color='crimson', linestyle='--', linewidth=2,
           label=f'MLE = {lambda_analytical:.4f}')
ax.axvline(x=true_lambda, color='black', linestyle='-', linewidth=1.5,
           label=f'True lambda = {true_lambda}')
ax.set_title('Log-likelihood function — exponential distribution')
ax.set_xlabel('lambda')
ax.set_ylabel('Log-likelihood')
ax.legend()
plt.tight_layout()
plt.savefig('log_likelihood.png', dpi=150, bbox_inches='tight')
plt.show()

print('MLE is the lambda value at the peak of the log-likelihood curve.')
print('The closer the MLE is to the true lambda, the better the estimation.')

In [ ]:
# MLE for normal distribution — two parameters simultaneously
# MLE estimates: mu_hat = sample mean, sigma_hat = sample std (ddof=0)
house_sample = np.random.choice(house_val, size=500, replace=True)

mu_mle, sigma_mle = stats.norm.fit(house_sample)

print('--- MLE for normal distribution (two parameters) ---')
print(f'MLE mu (sample mean):      {mu_mle:.4f}')
print(f'Analytical mean:           {house_sample.mean():.4f}')
print()
print(f'MLE sigma (sample std):    {sigma_mle:.4f}')
print(f'Sample std (ddof=0):       {house_sample.std(ddof=0):.4f}')
print(f'Sample std (ddof=1):       {house_sample.std(ddof=1):.4f}')
print()
print('MLE for sigma uses ddof=0 — it is biased.')
print('This is the bias-variance tradeoff from Section 1 reappearing.')
print('MLE is not always unbiased — it minimizes MSE, not bias specifically.')

---
## 6. Estimation in practice

### Margin of error

The margin of error is half the width of a confidence interval:

$$ME = z_{\alpha/2} \cdot \frac{\sigma}{\sqrt{n}}$$

### Sample size calculation

How large a sample do you need to achieve a desired margin of error $E$?

$$n = \left(\frac{z_{\alpha/2} \cdot \sigma}{E}\right)^2$$

This is the formula behind every survey sample size decision and A/B test power calculation.

In [ ]:
# Sample size calculation
sigma_est    = income.std()  # use population std as estimate
confidence   = 0.95
z            = stats.norm.ppf((1 + confidence) / 2)

desired_mes  = [0.5, 0.2, 0.1, 0.05]

print('--- Sample size required for desired margin of error ---')
print(f'Population std (sigma): {sigma_est:.4f}')
print(f'Confidence level:       {confidence*100:.0f}%')
print(f'z-value:                {z:.4f}')
print()
print(f'{"Desired ME":<15} {"Required n":<15} {"4x rule check"}')
print('-' * 50)
prev_n = None
for me in desired_mes:
    n_required = int(np.ceil((z * sigma_est / me) ** 2))
    ratio      = f'{n_required/prev_n:.1f}x' if prev_n else '-'
    print(f'{me:<15.2f} {n_required:<15,} {ratio}')
    prev_n = n_required
print()
print('Halving the margin of error requires ~4x the sample size (sqrt(n) relationship).')

In [ ]:
# Estimating difference between two groups — foundation of A/B testing
# High income vs low income neighborhoods: is mean house value different?
high_income_val = housing[housing['MedInc'] >= 5.0]['MedHouseVal'].values
low_income_val  = housing[housing['MedInc'] <  3.0]['MedHouseVal'].values

# Point estimate of difference
diff_observed = high_income_val.mean() - low_income_val.mean()

# Bootstrap CI for the difference
n_boot     = 5000
boot_diffs = []
for _ in range(n_boot):
    hi_boot = np.random.choice(high_income_val, size=len(high_income_val), replace=True)
    lo_boot = np.random.choice(low_income_val,  size=len(low_income_val),  replace=True)
    boot_diffs.append(hi_boot.mean() - lo_boot.mean())

boot_diffs = np.array(boot_diffs)
lo_diff    = np.percentile(boot_diffs, 2.5)
hi_diff    = np.percentile(boot_diffs, 97.5)

print('--- Difference in mean house value: high vs low income ---')
print(f'High income mean: {high_income_val.mean():.4f} ($100,000s)')
print(f'Low income mean:  {low_income_val.mean():.4f} ($100,000s)')
print(f'Observed difference: {diff_observed:.4f} ($100,000s = ${diff_observed*100:,.0f})')
print(f'95% Bootstrap CI:    ({lo_diff:.4f}, {hi_diff:.4f})')
print()
if lo_diff > 0:
    print('The CI does not include zero — the difference is statistically significant.')
    print('High income neighborhoods have meaningfully higher house values.')
else:
    print('The CI includes zero — cannot conclude the difference is statistically significant.')

In [ ]:
# Utility functions — reusable in production

def estimation_report(sample: np.ndarray, label: str = 'variable',
                       confidence: float = 0.95, n_boot: int = 5000) -> pd.DataFrame:
    """
    Full estimation report: point estimates, SE, analytical CI, bootstrap CI.
    Production-safe: uses only numpy and scipy.
    """
    n    = len(sample)
    mean = sample.mean()
    med  = np.median(sample)
    se   = stats.sem(sample)

    lo_t,    hi_t    = stats.t.interval(confidence, df=n-1, loc=mean, scale=se)
    lo_boot, hi_boot, _ = bootstrap_ci(sample, np.mean, n_boot=n_boot, confidence=confidence)

    print(f'--- Estimation report: {label} (n={n}) ---')
    print(f'Mean:                {mean:.4f}')
    print(f'Median:              {med:.4f}')
    print(f'Std dev:             {sample.std(ddof=1):.4f}')
    print(f'Standard error:      {se:.4f}')
    print(f'{int(confidence*100)}% CI (t-interval):   ({lo_t:.4f}, {hi_t:.4f})  width = {hi_t-lo_t:.4f}')
    print(f'{int(confidence*100)}% CI (bootstrap):    ({lo_boot:.4f}, {hi_boot:.4f})  width = {hi_boot-lo_boot:.4f}')
    print(f'Margin of error:     {(hi_t-lo_t)/2:.4f}')


estimation_report(np.random.choice(income, size=200, replace=True), 'MedInc (n=200)')
print()
estimation_report(np.random.choice(total_bill, size=50, replace=True), 'Total bill (n=50)')

---
## Summary

| Concept | Formula | Production tool |
|---|---|---|
| Standard error | $\sigma / \sqrt{n}$ | `scipy.stats.sem(sample)` |
| t-interval | $\bar{x} \pm t \cdot SE$ | `scipy.stats.t.interval(...)` |
| Bootstrap CI | Percentiles of bootstrap distribution | `bootstrap_ci()` above |
| MLE | $\arg\max_\theta \sum \log f(x_i; \theta)$ | `scipy.stats.dist.fit(data)` |
| Sample size | $(z \cdot \sigma / E)^2$ | Custom calculation |

**Key concepts to internalize:**
- MSE = Bias² + Variance — the fundamental tradeoff in estimation and ML
- 95% CI means 95% of such intervals contain the truth — not 95% probability for this interval
- Bootstrap gives CI for any statistic without a formula — use it as the default
- Halving margin of error requires 4x the sample size
- MLE is what `dist.fit()` does — it finds parameters maximizing log-likelihood

**Next:** Chapter 9 — Hypothesis Testing